# Cycle 0 - Baseline naïve

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import mlflow
import notebooks.utils as utils
from datetime import datetime as dt

In [3]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")

In [4]:
dataset = pd.read_csv('../data/rafined/dataset.csv', sep=',')

In [5]:
dataset.drop(columns=['Area ISO Code', 'Area ISO Alpha-2 Code', 'Area ISO Alpha-3 Code'], inplace=True)

In [6]:
X_train, X_test, y_train, y_test = utils.test_train_time_split(dataset)

Année de césure calculée : 2009
Proportion réelle train : 78.73% (20349 lignes)
Proportion réelle test  : 21.27% (5499 lignes)


In [7]:
mlflow.set_experiment("Agriculture_optimization")
mlflow.end_run()
mlflow.set_system_metrics_sampling_interval(1)

2026/11/19 14:21:33 INFO mlflow.tracking.fluent: Experiment with name 'Agriculture_optimization' does not exist. Creating a new experiment.


In [8]:
import datetime

# Baseline naïve : on prédit le rendement médian historique du couple (pays, culture)
train_df = X_train.copy()
train_df['hg/ha'] = y_train

# médiane du rendement par couple (pays, culture), calculée sur le train uniquement
mediane_pays_culture = train_df.groupby(['Area', 'Item'])['hg/ha'].median()

# pour chaque ligne du test, on récupère la médiane de son couple (pays, culture)
pred_baseline = X_test.join(mediane_pays_culture.rename('pred'), on=['Area', 'Item'])['pred']

# couples absents du train : médiane de la culture, tous pays confondus
mediane_culture = train_df.groupby('Item')['hg/ha'].median()
n_sans_historique = pred_baseline.isna().sum()
pred_baseline = pred_baseline.fillna(X_test['Item'].map(mediane_culture))

evaluation = utils.evaluate(y_test, pred_baseline)

print(f"Lignes test sans historique : {n_sans_historique}")
print(f"R²   : {evaluation['r2']:.4f}")
print(f"RMSE : {evaluation['rmse']:.0f} hg/ha")
print(f"MAE  : {evaluation['mae']:.0f} hg/ha")

with mlflow.start_run(run_name="Cycle 0 - Baseline médiane pays x culture", log_system_metrics=True):
    mlflow.set_tags({
        'date': dt.now().strftime("%Y-%m-%d"),
        'cycle': '0_baseline',
        'model_type': 'baseline_naive',
        'mlflow.note.content': (
            "Baseline naïve : prédit la médiane historique (train) du rendement "
            "pour chaque couple pays x culture. Référence à battre par les modèles."
        ),
    })
    mlflow.log_params({
        'split': 'temporel',
        'split_year': int(X_test['Year'].min()),
        'strategy': 'mediane_pays_culture',
        'fallback': 'mediane_culture',
    })
    mlflow.log_metrics(evaluation)

Lignes test sans historique : 39
R²   : 0.8364
RMSE : 32646 hg/ha
MAE  : 16461 hg/ha


2026/11/19 14:21:34 INFO mlflow.system_metrics.system_metrics_monitor: Skip logging GPU metrics. Set logger level to DEBUG for more details.
2026/11/19 14:21:34 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.
2026/11/19 14:21:34 INFO mlflow.system_metrics.system_metrics_monitor: Stopping system metrics monitoring...


🏃 View run Cycle 0 - Baseline médiane pays x culture at: http://127.0.0.1:5000/#/experiments/1/runs/b1272809d1e04369af36ae3f3ae6f826
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


2026/11/19 14:21:34 INFO mlflow.system_metrics.system_metrics_monitor: Successfully terminated system metrics monitoring!


## Conclusion du cycle 0 : baseline naïve

Sans machine learning, on prédit pour chaque couple (pays, culture) son rendement médian observé sur les années avant 2009. Les conditions de l'année (pluie, température, pesticides) ne sont pas prises en compte.

**Résultats sur le jeu de test (2009 à 2013, split temporel)** :

| Métrique | Valeur |
|---|---|
| R² | 0.836 |
| RMSE | 32 646 hg/ha |
| MAE | 16 461 hg/ha |


- Savoir simplement *quel pays* cultive *quelle culture* explique déjà 84 % de la variance du rendement : la culture (une pomme de terre produit beaucoup plus de kilos par hectare que du blé) et le pays (sol, climat moyen, niveau de mécanisation) sont les facteurs dominants.
- En moyenne, la prédiction se trompe d'environ 16 500 hg/ha, soit 1,6 tonne par hectare. La RMSE, deux fois plus élevée que la MAE, montre que certaines erreurs sont importantes : années exceptionnelles, ou forte progression des rendements dans certains pays depuis les années 1990, que la médiane historique ne capte pas. Ce second effet est net : la baseline sous-estime le rendement réel dans 72 % des cas sur la période de test, car les rendements ont globalement augmenté depuis 1990.